# Trajectory annotation

**Linking every trajectory point to the conditions the moving object passed through.**

Movement data and context data are observed at different resolutions, and movement is far denser. AIS messages arrive within minutes; weather is reported hourly, from a handful of coastal stations. So a vessel's position at 14:37, halfway between two stations, has no wind measurement of its own.

Annotation supplies one, in two stages:

| | | |
|---|---|---|
| **1. Spatial** | across stations, one epoch at a time | *what was the wind here, at that hour?* |
| **2. Temporal** | through time, at that position | *what was the wind here, at that minute?* |

Each stage has candidate methods, and each is compared by cross-validation.

The method applies to any moving object — vessels, aircraft, vehicles, wildlife, pedestrians — and to any context measured at fixed stations. This notebook works through it on real AIS and weather data from the western English Channel, February 2016; the maritime data is one application, not the scope. Every plot is interactive &mdash; hover for values, zoom, and click legend entries to mute a series. They need a live kernel, so they appear blank in a static render such as GitHub's notebook preview.

---

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import hvplot.pandas  # noqa: F401  -- adds .hvplot to DataFrames
import holoviews as hv

hv.extension("bokeh")

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
DATA = REPO / "datasets"

from catraj.annotation import (
    Nearest, IDW, TIN, CloughTocher, RBF, Kriging, to_surface,
    DLT, PiecewisePolynomial, Spline,
    evaluate_spatial, evaluate_temporal,
    annotate, annotate_points, decode_compass,
    parse_data_dictionary, describe_columns,
)

# Every table, figure and annotated track this notebook produces is written
# here, beside the notebook, so results outlive the kernel session.
RESULTS = Path.cwd() / "results_trajectory_annotation"
RESULTS.mkdir(exist_ok=True)


def save(obj, name, note=""):
    """Write a result to RESULTS, choosing the format from its type.

    DataFrames become CSV, GeoDataFrames GeoPackage, and holoviews objects
    standalone HTML that opens in any browser with no kernel and no network.
    """
    if isinstance(obj, gpd.GeoDataFrame):
        path = RESULTS / (name + ".gpkg")
        obj.to_file(path, driver="GPKG")
    elif isinstance(obj, pd.DataFrame):
        path = RESULTS / (name + ".csv")
        obj.to_csv(path, index=False)
    elif isinstance(obj, pd.Series):
        path = RESULTS / (name + ".csv")
        obj.to_frame().to_csv(path)
    elif isinstance(obj, str):
        path = RESULTS / (name + ".txt")
        path.write_text(obj, encoding="utf-8")
    else:                                    # holoviews / bokeh figure
        path = RESULTS / (name + ".html")
        hv.save(obj, path, backend="bokeh")
    print(f"  saved  {path.name:<42} {path.stat().st_size / 1024:8.1f} KB  {note}")
    return obj


print(f"ready - results go to {RESULTS.name}/")

ready - results go to results_trajectory_annotation/


Results are written to **`results_trajectory_annotation/`**, beside this notebook.
Each file is named for the step that produced it, so the folder reads as a record of the
analysis:

| prefix | what it holds |
|---|---|
| `01_...` | the input data, as loaded |
| `02_...` | stage 1 &mdash; spatial interpolation and its cross-validation |
| `03_...` | stage 2 &mdash; temporal interpolation and its cross-validation |
| `04_...` | the annotated trajectory |

Figures are saved as **standalone HTML** &mdash; those open in any browser with no kernel
and no network, unlike the plots inline in this notebook.

---
## 1. The data

Two matched shapefiles covering **exactly the same 24 hours**, 2016-02-05 01:00 to 2016-02-06 01:00 UTC:

- **`Weather_Observation_UTM.shp`** &mdash; 16 coastal stations, hourly
- **`AIS_DLT_UTM.shp`** &mdash; 106,894 vessel positions

Both are projected to **UTM zone 30N**, so distances are in metres. That matters: spatial interpolation needs a metric space, and degrees are not one &mdash; at 48&deg;N a degree of longitude spans about 74 km against 111 km for a degree of latitude.

For mapping we also keep a Web Mercator copy, which is what tile basemaps use.

In [2]:
weather = gpd.read_file(DATA / "context" / "Weather" / "Weather_Observation_UTM.shp")
weather["x"] = weather.geometry.x          # UTM metres -- used for interpolation
weather["y"] = weather.geometry.y
weather["time"] = pd.to_datetime(weather.local_time, unit="s")
weather["station"] = weather.id_station.astype(str)

# Web Mercator copy, for plotting on a basemap
mercator = weather.to_crs(3857)
weather["mx"] = mercator.geometry.x
weather["my"] = mercator.geometry.y

print(f"{len(weather)} observations · {weather.station.nunique()} stations "
      f"· {weather.local_time.nunique()} epochs")
print(f"{weather.time.min()} to {weather.time.max()} UTC · CRS {weather.crs.name}")
save(weather.drop(columns="geometry"), "01_weather_observations",
     "16 stations x 25 hourly epochs")

weather[["station", "time", "x", "y", "T", "Ff", "id_windDir", "P"]].head()

C:\ProgramData\miniconda3\envs\trajctx\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


400 observations · 16 stations · 25 epochs
2016-02-05 01:00:00 to 2016-02-06 01:00:00 UTC · CRS WGS 84 / UTM zone 30N
  saved  01_weather_observations.csv                    71.6 KB  16 stations x 25 hourly epochs


,station,time,x,y,T,Ff,id_windDir,P
0,7100,2016-02-05 01:00:00,347977.124009,5.370951e+06,10.1,5.0,12,772.6
1,3803,2016-02-05 01:00:00,263112.557063,5.534591e+06,10.6,9.0,11,769.9
2,7300,2016-02-05 01:00:00,550969.192938,5.172043e+06,10.4,2.0,11,774.5
3,7207,2016-02-05 01:00:00,483617.703038,5.238526e+06,10.6,6.0,12,774.1
4,7020,2016-02-05 01:00:00,576894.985181,5.507668e+06,10.5,6.0,12,772.4


### What are `Ff`, `Td`, `id_windDir`?

Nothing in those names says what they measure. Both archives ship a plain-text
description of their fields, and `parse_data_dictionary` reads it — so the
columns can be shown with their meaning and units instead of guessed at.

This is optional. A dataset whose column names already speak for themselves
needs none of it.

In [3]:
weather_fields = parse_data_dictionary(
    DATA / "context" / "Weather" / "Weather_parameters.txt")
ais_fields = parse_data_dictionary(DATA / "AIS" / "AIS_parameters.txt")

print(f"{len(weather_fields)} weather fields, {len(ais_fields)} AIS fields "
      f"described\n")

# Shapefiles truncate names to ten characters, so `id_windDirection` in the
# dictionary has to match a column called `id_windDir`. describe_columns
# handles that, and falls back to the bare name when nothing is known.
labels = describe_columns(
    [c for c in weather.columns if c != "geometry"], weather_fields)

pd.DataFrame(
    [{"column": c, "means": labels[c].split(" - ", 1)[-1]}
     for c in ["T", "P", "U", "Ff", "id_windDir", "VV", "Td", "local_time"]]
)

15 weather fields, 24 AIS fields described



,column,means
0,T,Air temperature (°C)
1,P,Atmospheric pressure (mm of mercury)
2,U,Relative humidity (%)
3,Ff,Mean wind speed (meters per seconds)
4,id_windDir,Mean wind direction (compass point)
5,VV,Horizontal visibility (km)
6,Td,Dewpoint temperature
7,local_time,"unix timestamp - number of seconds since ""01/0..."


In [4]:
ais = gpd.read_file(DATA / "AIS" / "AIS_DLT_UTM.shp")
ais["time"] = pd.to_datetime(ais.t, unit="s")

print(f"{len(ais):,} positions · {ais.sourcemmsi.nunique()} vessels")
print(f"{ais.time.min()} to {ais.time.max()} UTC")
save(ais.vesselType.value_counts(), "01_ais_vessel_types", "fleet composition")

ais.vesselType.value_counts().head(6)

106,894 positions · 136 vessels
2016-02-05 01:00:02 to 2016-02-06 01:00:00 UTC
  saved  01_ais_vessel_types.csv                         0.2 KB  fleet composition


vesselType
Passenger    25906
Reserved     25447
Fishing      16513
Tug           8548
Cargo         8201
Tanker        7378
Name: count, dtype: int64

### The study area

Weather stations against the vessel traffic they have to describe. The stations ring the coast; the shipping lanes run well offshore, which is why interpolation &mdash; not lookup &mdash; is needed.

In [5]:
ais_merc = ais.iloc[::60].to_crs(3857)
traffic = pd.DataFrame({"mx": ais_merc.geometry.x, "my": ais_merc.geometry.y})

study_area = traffic.hvplot.points(
    x="mx", y="my", color="#0969da", size=3, alpha=0.25,
    tiles="OSM", data_aspect=1, frame_width=620, frame_height=520,
    xaxis=None, yaxis=None, title="AIS traffic and weather stations",
) * weather.drop_duplicates("station").hvplot.points(
    x="mx", y="my", color="#e16f24", size=140, marker="triangle",
    line_color="white", hover_cols=["station"],
)

save(study_area, "01_study_area_map", "traffic and stations on OSM")
study_area

  saved  01_study_area_map.html                         47.2 KB  traffic and stations on OSM


:Overlay
   .Tiles.I   :Tiles   [x,y]
   .Points.I  :Points   [mx,my]
   .Points.II :Points   [mx,my]   (station)

### The resolution gap

One vessel's positions against the weather epochs. Every AIS dot between two weather ticks has no context value of its own &mdash; that is the gap annotation fills.

In [6]:
busiest = ais.sourcemmsi.value_counts().index[0]
one = ais[ais.sourcemmsi == busiest].sort_values("t")
epochs = pd.to_datetime(sorted(weather.local_time.unique()), unit="s")

gap = pd.concat([
    pd.DataFrame({"time": one.time, "row": "movement", "kind": "AIS position"}),
    pd.DataFrame({"time": epochs, "row": "context", "kind": "weather epoch"}),
])

print(f"{len(one):,} positions against {len(epochs)} context epochs")

gap.hvplot.scatter(
    x="time", y="row", by="kind", size=40, alpha=0.7,
    frame_width=740, frame_height=170, ylabel="",
    title=f"vessel {busiest} - one day",
).opts(legend_position="top_right")

9,196 positions against 25 context epochs


:NdOverlay   [kind]
   :Scatter   [time]   (row)

---
## 2. Stage 1 &mdash; spatial interpolation

Context is measured at stations, never where the moving object is. For each epoch we interpolate the station values **directly to the query position** &mdash; no raster is built.

Take a single epoch and look at what the stations recorded — here **mean wind speed**, the `Ff` column. Hover a marker for its identifier and readings.

In [7]:
epoch = sorted(weather.local_time.unique())[12]   # midday-ish
snap = weather[weather.local_time == epoch]

station_map = snap.hvplot.points(
    x="mx", y="my", c="Ff", cmap="viridis", size=200, line_color="white",
    hover_cols=["station", "T", "P"], clabel="wind speed (m/s)",
    tiles="CartoLight", data_aspect=1, frame_width=560, frame_height=480,
    xaxis=None, yaxis=None,
    title=f"{labels['Ff']} at {pd.to_datetime(epoch, unit='s')} UTC",
)

save(snap.drop(columns="geometry"), "02_epoch_snapshot", "the epoch used below")
save(station_map, "02_station_map")
station_map

  saved  02_epoch_snapshot.csv                           3.0 KB  the epoch used below


  saved  02_station_map.html                            23.1 KB  


:Overlay
   .Tiles.I  :Tiles   [x,y]
   .Points.I :Points   [mx,my]   (Ff,station,T,P)

### The candidate methods

| | |
|---|---|
| **Nearest** | piecewise constant &mdash; an error floor |
| **IDW** | inverse distance weighting, `w = 1/d^p` |
| **TIN** | linear on a Delaunay triangulation |
| **CloughTocher** | cubic C&sup1; on the same triangulation |
| **RBF** | thin-plate spline, smooth |
| **Kriging** | Gaussian process &mdash; also returns uncertainty |

All are *exact* at the stations except Kriging with a nugget: a query landing on a station returns that station's value.

In [8]:
centre = (snap.x.mean(), snap.y.mean())

comparison = pd.DataFrame([
    {"method": M.__name__,
     "estimate_at_centroid": M().fit(snap.x, snap.y, snap.Ff)(*centre)}
    for M in (Nearest, IDW, TIN, CloughTocher, RBF, Kriging)
])

fit = IDW().fit(snap.x, snap.y, snap.Ff)
print(f"IDW at station 1: {fit(snap.x.iloc[0], snap.y.iloc[0]):.3f}"
      f"  (observed {snap.Ff.iloc[0]:.3f})  -- exact at the stations\n")

save(comparison, "02_methods_at_centroid", "one epoch, six methods")
comparison.round(3)

IDW at station 1: 10.000  (observed 10.000)  -- exact at the stations

  saved  02_methods_at_centroid.csv                      0.2 KB  one epoch, six methods


,method,estimate_at_centroid
0,Nearest,7.000
1,IDW,7.006
2,TIN,6.802
3,CloughTocher,6.720
4,RBF,6.958
5,Kriging,6.964


### Surfaces, to see the difference

Annotation samples at points and never needs a raster, but `to_surface` builds one &mdash; useful for seeing how the methods differ. The panels share axes, so zooming one zooms all six.

Watch the triangulation methods: they return **nothing outside the convex hull** of the stations. That is deliberate. They have no basis for a value there, and admitting the gap is better than inventing one.

In [9]:
pad = 40_000
bounds = (snap.x.min() - pad, snap.y.min() - pad,
          snap.x.max() + pad, snap.y.max() + pad)

stations_utm = snap.hvplot.points(
    x="x", y="y", color="white", size=40, line_color="black",
    hover_cols=["station", "Ff"],
)

panels = []
for m in [Nearest(), IDW(power=2), TIN(), CloughTocher(), RBF(), Kriging()]:
    grid, xs, ys = to_surface(m.fit(snap.x, snap.y, snap.Ff), bounds, 6000)
    name = type(m).__name__
    img = hv.Image((xs, ys, grid), kdims=["x", "y"], vdims="wind speed").opts(
        cmap="viridis", frame_width=250, frame_height=230, colorbar=False,
        xaxis=None, yaxis=None,
        title=f"{name}  ({100 * np.isfinite(grid).mean():.0f}% covered)",
    )
    panels.append(img * stations_utm)
    pd.DataFrame(grid, index=ys, columns=xs).to_csv(
        RESULTS / f"02_surface_{name.lower()}.csv")

surfaces = hv.Layout(panels).cols(3).opts(shared_axes=True)
print("  saved  six interpolated grids as 02_surface_*.csv")
save(surfaces, "02_surfaces", "six interpolators, one epoch")
surfaces

  saved  six interpolated grids as 02_surface_*.csv


  saved  02_surfaces.html                              520.1 KB  six interpolators, one epoch


:Layout
   .Overlay.I   :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)
   .Overlay.II  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)
   .Overlay.III :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)
   .Overlay.IV  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)
   .Overlay.V   :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)
   .Overlay.VI  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,Ff)

### Which method is best? Leave-one-station-out

Hold out one station, predict its value from the other 15, compare against what it actually observed. Repeat for every station, at every epoch.

This is the spatial analogue of k-fold, with **stations rather than time** as the folds.

Two things get reported that a bare error figure would hide:

- **unreachable predictions** &mdash; a held-out station may fall outside the hull of the others. Corner stations often do, and the triangulation methods cannot reach them. A method that fails at edge stations is worse in a way its error alone does not show.
- **station count** &mdash; every fit uses one fewer station than exist. With 16, that is thin for kriging.

In [10]:
table = evaluate_spatial(weather.x, weather.y, weather.Ff, weather.local_time)
print(table.summary())

df = table.to_frame()

save(table.summary(), "02_spatial_cv_summary")
save(df, "02_spatial_cv_by_epoch", "every method at every epoch")
ranked = (df.groupby("method")[["rmse", "mae", "me", "max_ae", "r2", "unreachable"]]
            .mean().sort_values("rmse"))
save(ranked.reset_index(), "02_spatial_cv_ranked", "averaged over epochs")

ranked.round(3)

leave-one-station-out over 25 epoch(s), 16-16 stations
  tin            mean rmse=1.588   unreachable=150
  clough_tocher  mean rmse=1.733   unreachable=150
  nearest        mean rmse=2.538
  idw            mean rmse=2.545
  kriging        mean rmse=2.698
  rbf            mean rmse=2.775
  saved  02_spatial_cv_summary.txt                       0.3 KB  
  saved  02_spatial_cv_by_epoch.csv                     17.0 KB  every method at every epoch
  saved  02_spatial_cv_ranked.csv                        0.6 KB  averaged over epochs


,rmse,mae,me,max_ae,r2,unreachable
method,,,,,,
tin,1.588,1.305,0.274,2.894,-0.053,6.0
clough_tocher,1.733,1.389,0.196,3.394,-0.271,6.0
nearest,2.538,1.940,-0.255,5.960,0.085,0.0
idw,2.545,2.035,-0.140,5.716,0.105,0.0
kriging,2.698,2.190,-0.089,5.822,-0.017,0.0
rbf,2.775,2.178,-0.159,6.416,-0.177,0.0


Five metrics are reported together because no one of them is sufficient:

- **RMSE** and **MAE** are both magnitude-only. A model that consistently under-estimates by 2 m/s scores identically to one scattering &plusmn;2 m/s symmetrically.
- **ME** is signed, so it is the only one that reveals *bias*.
- **Max|e|** exposes rare catastrophic points that RMSE averages away.
- The **gap between RMSE and MAE** is itself diagnostic &mdash; a large gap means a few big errors, a small gap means they are spread evenly.

Click a method in the legend to mute it.

In [11]:
df["time"] = pd.to_datetime(df.epoch, unit="s")

cv_by_epoch = df.hvplot.line(
    x="time", y="rmse", by="method", muted_alpha=0.1,
    frame_width=700, frame_height=300, ylabel="RMSE (m/s)",
    title="leave-one-station-out error, by epoch",
).opts(legend_position="right")

save(cv_by_epoch, "02_spatial_cv_by_epoch_plot")
cv_by_epoch

  saved  02_spatial_cv_by_epoch_plot.html               34.2 KB  


:NdOverlay   [method]
   :Curve   [time]   (rmse)

That variation is why **per-epoch selection** exists as an option. A calm, smooth field and a storm front are not best served by the same interpolator.

It is not the default, though. The method can then switch between consecutive hours, and a value that moves because the *interpolator* changed rather than the weather is an artefact &mdash; one the temporal stage will faithfully reproduce as roughness.

In [12]:
overall = table.select()
per_epoch = table.select(per_epoch=True)

print("one method for all epochs :", overall)
print("\nper-epoch picks:")
print(pd.Series(list(per_epoch.values())).value_counts().to_string())

save(pd.DataFrame({"epoch": list(per_epoch), "method": list(per_epoch.values())})
       .assign(time=lambda d: pd.to_datetime(d.epoch, unit="s")),
     "02_spatial_method_per_epoch", f"the single-method choice was {overall}")

one method for all epochs : tin

per-epoch picks:
tin              18
clough_tocher     7
  saved  02_spatial_method_per_epoch.csv                 1.0 KB  the single-method choice was tin


,epoch,method,time
0,1454634000,tin,2016-02-05 01:00:00
1,1454637600,tin,2016-02-05 02:00:00
2,1454641200,tin,2016-02-05 03:00:00
3,1454644800,tin,2016-02-05 04:00:00
4,1454648400,tin,2016-02-05 05:00:00
5,1454652000,tin,2016-02-05 06:00:00
6,1454655600,clough_tocher,2016-02-05 07:00:00
7,1454659200,clough_tocher,2016-02-05 08:00:00
8,1454662800,tin,2016-02-05 09:00:00
9,1454666400,clough_tocher,2016-02-05 10:00:00


### Wind direction is different &mdash; it is a bearing

0&deg; and 360&deg; are the same direction. Interpolated as ordinary numbers, stations reading 350&deg; and 10&deg; average to **180&deg;** &mdash; the exact opposite heading.

So bearings are interpolated **on the unit circle**: convert each to a vector, interpolate `cos` and `sin` independently, recover with `atan2`.

**The caller declares which variables are circular.** Nothing is inferred from a column name.

This dataset also stores direction as a **compass-point code**, not degrees &mdash; `degrees = code x 22.5` on a 16-point rose. Interpolating the codes would be meaningless; they are categorical labels, and a step from code 5 to code 13 is not a rotation.

In [13]:
# The dictionary is what tells us this column is a *compass point*, not a
# bearing in degrees -- the column name alone would not.
print(labels["id_windDir"], "\n")

weather["wind_dir"] = decode_compass(weather.id_windDir)
print("codes  :", sorted(weather.id_windDir.unique())[:8])
print("degrees:", sorted(weather.wind_dir.dropna().unique())[:8])

# A worked example that crosses north, where the two treatments diverge.
xs_n = np.array([0.0, 1e5, 0.0])
ys_n = np.array([0.0, 0.0, 1e5])
bearings = np.array([350.0, 10.0, 0.0])
q = (3e4, 3e4)

plain = IDW(circular=False).fit(xs_n, ys_n, bearings)(*q)
circ = IDW(circular=True).fit(xs_n, ys_n, bearings)(*q)
print(f"\nstations reading {bearings.tolist()} degrees:")
print(f"  as plain numbers : {plain:6.1f}   <- points south, wrong")
print(f"  as bearings      : {circ:6.1f}   <- correct")

id_windDirection - Mean wind direction (compass point) 

codes  : [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
degrees: [np.float64(112.5), np.float64(135.0), np.float64(157.5), np.float64(180.0), np.float64(202.5), np.float64(225.0), np.float64(247.5), np.float64(270.0)]

stations reading [350.0, 10.0, 0.0] degrees:
  as plain numbers :  217.9   <- points south, wrong
  as bearings      :  355.7   <- correct


---
## 3. Stage 2 &mdash; temporal interpolation

Stage 1 gave us a value at the object's position for every epoch. Now we fit a curve through that series and read it at the exact timestamp.

Three candidates, from [Mehri2023] &sect;4.1:

| | | |
|---|---|---|
| **DLT** | Eq. 2 | straight line between consecutive epochs |
| **PiecewisePolynomial** | Eq. 3 | contiguous least-squares windows, degrees 2&ndash;7 |
| **Spline** | Eq. 4 | one cubic curve through every observation |

DLT assumes the variable changes linearly across the gap. Real environmental variables curve, and the coarser the observations the worse a straight line fits &mdash; which is what motivates the two nonlinear methods.

Overlaying them shows where they actually *disagree*. Zoom into a single gap.

In [14]:
sid = weather.station.iloc[0]
series = weather[weather.station == sid].sort_values("local_time")
t, v = series.local_time.to_numpy(float), series.Ff.to_numpy(float)
dense = np.linspace(t.min(), t.max(), 600)

fits = {"DLT": DLT(),
        "PiecewisePolynomial (deg 2)": PiecewisePolynomial(degree=2),
        "Spline": Spline()}

curves = pd.concat([
    pd.DataFrame({"time": pd.to_datetime(dense, unit="s"),
                  "wind speed": m.fit(t, v)(dense), "method": name})
    for name, m in fits.items()
])
observed = pd.DataFrame({"time": pd.to_datetime(t, unit="s"), "wind speed": v})

method_curves = curves.hvplot.line(
    x="time", y="wind speed", by="method", muted_alpha=0.15,
    frame_width=700, frame_height=320, ylabel="wind speed (m/s)",
    title=f"station {sid} - the same 25 observations, three methods",
) * observed.hvplot.scatter(
    x="time", y="wind speed", color="#e16f24", size=70, line_color="white",
)

save(curves, "03_temporal_fitted_curves", "three methods, densely sampled")
save(method_curves, "03_temporal_methods_plot")
method_curves

  saved  03_temporal_fitted_curves.csv                 107.2 KB  three methods, densely sampled


  saved  03_temporal_methods_plot.html                  95.3 KB  


:Overlay
   .NdOverlay.I :NdOverlay   [method]
      :Curve   [time]   (wind speed)
   .Scatter.I   :Scatter   [time]   (wind speed)

Zoom in on the piecewise polynomial and you will see its curve **does not pass through the observations** &mdash; it passes *near* them.

That is not a bug. Eq. 3 partitions the interval into contiguous windows, and with `window = degree + 2` each sub-polynomial has `d+1` coefficients fitted to `d+2` points &mdash; a least-squares fit with one degree of freedom. A window of exactly `d+1` points would be forced through every one, leaving zero residual and making every degree score identically.

The spline, by contrast, is **interpolating**: it passes through every observation exactly, and matches slope and curvature at every knot. That difference is what the comparison is really measuring.

In [15]:
residuals = pd.DataFrame([
    {"method": name,
     "max_abs_residual_at_observations": float(np.abs(m.fit(t, v)(t) - v).max())}
    for name, m in [("PiecewisePolynomial", PiecewisePolynomial(degree=2)),
                    ("Spline", Spline())]
])

save(residuals, "03_interpolating_vs_least_squares",
     "zero means the curve passes through every observation")
residuals

  saved  03_interpolating_vs_least_squares.csv           0.1 KB  zero means the curve passes through every observation


,method,max_abs_residual_at_observations
0,PiecewisePolynomial,1.35
1,Spline,0.00


### Which method is best? k-fold

Observations are partitioned **randomly** into k sets. Each fold is held out in turn, the rest fits the model, the held-out fold is scored. Reported metrics are the mean over all k folds.

`k` does double duty: training size grows with k, so sweeping k = 2&hellip;10 varies both the fold structure *and* the number of fitting points.

**The first and last observation are always kept in training.** Random partitioning could otherwise hold one out, leaving the training set unable to span it &mdash; and the model would be extrapolating, not interpolating. Context observations bracket the trajectory by design, so extrapolation is prevented structurally rather than merely scored.

In [16]:
ttable = evaluate_temporal(t, v)
print(ttable.summary())

save(ttable.summary(), "03_temporal_cv_summary")
save(ttable.to_frame(), "03_temporal_cv_full", "every method at every k")

kfold_heatmap = ttable.to_frame().hvplot.heatmap(
    x="k", y="method", C="rmse", cmap="viridis_r",
    frame_width=520, frame_height=300, clabel="RMSE (m/s)",
    title="every method at every fold count - darker is better",
).opts(xrotation=0)

save(kfold_heatmap, "03_temporal_cv_heatmap")
kfold_heatmap

k-fold over k=2..10, 25 observations
  pp_deg2      best k=10  rmse=0.9298   MAE=0.7847  n_fit=23
  dlt          best k=5   rmse=0.9646   MAE=0.7667  n_fit=21
  pp_deg3      best k=2   rmse=0.9728   MAE=0.8037  n_fit=14
  pp_deg5      best k=3   rmse=1.309   MAE=1.142  n_fit=18
  spline       best k=3   rmse=1.314   MAE=1.054  n_fit=18
  pp_deg4      best k=3   rmse=1.32   MAE=1.105  n_fit=18
  pp_deg6      best k=4   rmse=1.446   MAE=1.313  n_fit=20
  pp_deg7      best k=4   rmse=1.482   MAE=1.286  n_fit=20
  saved  03_temporal_cv_summary.txt                      0.5 KB  
  saved  03_temporal_cv_full.csv                         7.9 KB  every method at every k


  saved  03_temporal_cv_heatmap.html                    22.8 KB  


:HeatMap   [k,method]   (rmse)

### Selection is deliberately a separate step

`evaluate_temporal` returns the table; the analyst chooses. The published study shows why that is not purely mechanical &mdash; for mean wind speed, DLT won on MAE while the spline won on RMSE, and the spline was chosen *because* RMSE weights the large errors that matter for this application.

`select()` returns the winner **already fitted to every observation**, ready to annotate with.

In [17]:
by_rmse, by_mae = ttable.best_row("rmse"), ttable.best_row("mae")
print(f"best by RMSE: {by_rmse.method} (k={by_rmse.k})")
print(f"best by MAE : {by_mae.method} (k={by_mae.k})")
if (by_rmse.method, by_rmse.k) != (by_mae.method, by_mae.k):
    print("\n-> they disagree; the choice is a judgement, not a default")

model = ttable.select()          # or ttable.select(method="spline")
print(f"\nselected and fitted: {model}")

save(pd.DataFrame([
    {"rule": "rmse", "method": by_rmse.method, "k": by_rmse.k,
     "score": by_rmse.metrics.rmse},
    {"rule": "mae", "method": by_mae.method, "k": by_mae.k,
     "score": by_mae.metrics.mae},
]), "03_temporal_selection", "the two rules, and what each chose")

best by RMSE: pp_deg2 (k=10)
best by MAE : pp_deg2 (k=4)

-> they disagree; the choice is a judgement, not a default

selected and fitted: PiecewisePolynomial(degree=2, window=4, step=3, circular=False)
  saved  03_temporal_selection.csv                       0.1 KB  the two rules, and what each chose


,rule,method,k,score
0,rmse,pp_deg2,10,0.929802
1,mae,pp_deg2,4,0.761063


---
## 4. Putting it together

`annotate_points` runs both stages: spatial across stations at each epoch, then temporal through the resulting series.

The stations table just needs four columns: `x`, `y`, `t`, `value` — whatever the source called them.

Every position is annotated — no sampling. Each epoch's spatial interpolator is fitted once and evaluated at all positions together, so the whole track costs a few seconds.

In [18]:
stations = pd.DataFrame({
    "x": weather.x, "y": weather.y,
    "t": weather.local_time.astype(float),
    "value": weather.Ff.astype(float),
})

track = one.copy()                     # every point, not a sample
track["wind_speed"] = annotate_points(
    stations,
    track.geometry.x.to_numpy(), track.geometry.y.to_numpy(),
    track.t.to_numpy(float),
    spatial_method=IDW(power=2), temporal_method=Spline(),
)

print(f"annotated {track.wind_speed.notna().sum()}/{len(track)} positions "
      f"of vessel {busiest}")
track[["time", "speedoverg", "wind_speed"]].head()

annotated 9196/9196 positions of vessel 234056000


,time,speedoverg,wind_speed
84768,2016-02-05 01:00:09,0.2,4.663151
84769,2016-02-05 01:00:20,0.2,4.660977
84770,2016-02-05 01:00:31,0.2,4.658811
84771,2016-02-05 01:00:41,0.1,4.656857
84772,2016-02-05 01:00:49,0.1,4.655306


The track on a basemap, coloured by the context annotation gave it. Hover a position to read the vessel's own speed alongside the wind speed it was moving through.

In [19]:
track_merc = track.to_crs(3857)
track_plot = track.assign(
    mx=track_merc.geometry.x, my=track_merc.geometry.y,
    hour=track.time.dt.strftime("%H:%M"),
)

track_map = track_plot.hvplot.points(
    x="mx", y="my", c="wind_speed", cmap="viridis", size=30,
    hover_cols=["hour", "speedoverg"], clabel="wind speed (m/s)",
    tiles="CartoLight", data_aspect=1, frame_width=560, frame_height=460,
    xaxis=None, yaxis=None, title=f"vessel {busiest}, annotated",
) * snap.hvplot.points(
    x="mx", y="my", color="#e16f24", marker="triangle", size=120,
    line_color="white", hover_cols=["station"],
)

save(track_map, "04_annotated_track_map", "on a basemap")
track_map

  saved  04_annotated_track_map.html                   373.3 KB  on a basemap


:Overlay
   .Tiles.I   :Tiles   [x,y]
   .Points.I  :Points   [mx,my]   (wind_speed,hour,speedoverg)
   .Points.II :Points   [mx,my]   (station)

The annotated series is smooth and follows the regional trend, but it is **not** the station mean &mdash; it tracks conditions along *this vessel's* path, which is the whole point.

In [20]:
station_mean = weather.groupby("time").Ff.mean().rename("station mean").reset_index()

along_path = track.hvplot.line(
    x="time", y="wind_speed", color="#1a7f37", line_width=2,
    frame_width=700, frame_height=280, ylabel="wind speed (m/s)",
    label="annotated along the track", title="context along the trajectory",
) * station_mean.hvplot.scatter(
    x="time", y="station mean", color="#e16f24", size=45,
    label="mean of all 16 stations",
)

save(along_path, "04_context_along_path")
along_path

  saved  04_context_along_path.html                    537.2 KB  


:Overlay
   .Curve.Annotated_along_the_track :Curve   [time]   (wind_speed)
   .Scatter.Mean_of_all_16_stations :Scatter   [time]   (station mean)

### With MovingPandas

`annotate` writes onto a trajectory's dataframe. Because context lives as an ordinary column, it survives the `.iloc` slicing that trajectory compression performs &mdash; so a compressed trajectory keeps the context of the points it retains.

In [21]:
import movingpandas as mpd

traj = mpd.Trajectory(track.set_index("time"), traj_id=str(busiest))
annotate(traj, stations, "wind_speed_mpd",
         spatial_method=IDW(power=2), temporal_method=Spline())

print(traj)
traj.df[["speedoverg", "wind_speed_mpd"]].head()

C:\ProgramData\miniconda3\envs\trajctx\Lib\site-packages\movingpandas\__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


Trajectory 234056000 (2016-02-05 01:00:09 to 2016-02-06 01:00:00) | Size: 9137 | Length: 108.5 km
Bounds: (370074.67603251676, 5331501.95155489, 395203.9805629485, 5358781.732028517)
LINESTRING (388949.2125785124 5340486.865935527, 388949.4490945294 5340486.416583046, 388950.0508639


,speedoverg,wind_speed_mpd
time,,
2016-02-05 01:00:09,0.2,1.127341e+16
2016-02-05 01:00:20,0.2,1.127333e+16
2016-02-05 01:00:31,0.2,1.127313e+16
2016-02-05 01:00:41,0.1,1.127308e+16
2016-02-05 01:00:49,0.1,1.127322e+16


---
## 5. Several variables at once

Each context variable gets its own choice of method &mdash; and `circular=True` where the variable is a bearing.

The panels share a time axis, so zooming one zooms them all: a wind shift can be read against the pressure change that accompanied it.

In [22]:
variables = {
    "wind_speed":  ("Ff", False, "m/s"),
    "temperature": ("T", False, "degC"),
    "pressure":    ("P", False, "mmHg"),
    "wind_dir":    ("wind_dir", True, "degrees"),   # a bearing
}

for name, (col, circular, _) in variables.items():
    st = stations.assign(value=weather[col].astype(float).to_numpy())
    track[name] = annotate_points(
        st, track.geometry.x.to_numpy(), track.geometry.y.to_numpy(),
        track.t.to_numpy(float),
        spatial_method=IDW(power=2, circular=circular),
        temporal_method=Spline(circular=circular),
        circular=circular,
    )

save(track.drop(columns="geometry"), "04_annotated_track_all_variables",
     "four context variables on one track")
save(gpd.GeoDataFrame(track, geometry="geometry", crs=ais.crs),
     "04_annotated_track", "same, with geometry -- opens in QGIS")

# `wind_dir` is a column we derived, so it is described by the source
# column it came from.
source_of = {"wind_dir": "id_windDir"}

print("annotated:")
for name, (col, _, unit) in variables.items():
    described = labels.get(source_of.get(col, col), col)
    print(f"  {name:12} {described}")

track[["time"] + list(variables)].head()

  saved  04_annotated_track_all_variables.csv         1489.1 KB  four context variables on one track


  saved  04_annotated_track.gpkg                      2052.0 KB  same, with geometry -- opens in QGIS
annotated:
  wind_speed   Ff - Mean wind speed (meters per seconds)
  temperature  T - Air temperature (°C)
  pressure     P - Atmospheric pressure (mm of mercury)
  wind_dir     id_windDirection - Mean wind direction (compass point)


,time,wind_speed,temperature,pressure,wind_dir
84768,2016-02-05 01:00:09,4.663151,9.957718,772.733810,262.674065
84769,2016-02-05 01:00:20,4.660977,9.957318,772.732847,262.596918
84770,2016-02-05 01:00:31,4.658811,9.956918,772.731877,262.519825
84771,2016-02-05 01:00:41,4.656857,9.956556,772.730989,262.449966
84772,2016-02-05 01:00:49,4.655306,9.956267,772.730273,262.394275


In [23]:
panels_all = hv.Layout([
    track.hvplot.line(
        x="time", y=name, color="#0969da",
        frame_width=680, frame_height=150, ylabel=f"{name} ({unit})",
    )
    for name, (_, _, unit) in variables.items()
]).cols(1).opts(shared_axes=True)

save(panels_all, "04_all_variables_plot", "linked time axes")
panels_all

  saved  04_all_variables_plot.html                   2083.9 KB  linked time axes


:Layout
   .Curve.I   :Curve   [time]   (wind_speed)
   .Curve.II  :Curve   [time]   (temperature)
   .Curve.III :Curve   [time]   (pressure)
   .Curve.IV  :Curve   [time]   (wind_dir)

---

## What was saved

Everything above is now on disk in `results_trajectory_annotation/`, so the analysis
survives the kernel. CSVs open anywhere, the GeoPackage opens in QGIS, and the HTML
figures stay interactive in a browser with no kernel and no network.


In [24]:
manifest = pd.DataFrame(
    [{"file": f.name, "kind": f.suffix.lstrip("."),
      "size_kb": round(f.stat().st_size / 1024, 1)}
     for f in sorted(RESULTS.iterdir()) if f.is_file()]
)
manifest.to_csv(RESULTS / "00_manifest.csv", index=False)

print(f"{len(manifest)} files, {manifest.size_kb.sum() / 1024:.1f} MB "
      f"in {RESULTS.name}/")
manifest

31 files, 8.2 MB in results_trajectory_annotation/


,file,kind,size_kb
0,00_manifest.csv,csv,1.1
1,01_ais_vessel_types.csv,csv,0.2
2,01_study_area_map.html,html,47.2
3,01_weather_observations.csv,csv,71.6
4,02_epoch_snapshot.csv,csv,3.0
5,02_methods_at_centroid.csv,csv,0.2
6,02_spatial_cv_by_epoch.csv,csv,17.0
7,02_spatial_cv_by_epoch_plot.html,html,34.2
8,02_spatial_cv_ranked.csv,csv,0.6
9,02_spatial_cv_summary.txt,txt,0.3
